# 01 · SQL exploration on DuckDB

Goal: build the DuckDB database and practice the queries the agent will have to write later.
Every question in this notebook is a candidate for the **evaluation set** (`evals/`), since we compute its correct answer ourselves.

In [ ]:
from demand_copilot.db import build_database, get_connection, describe_schema

# Creates data/processed/sales.duckdb from data/raw (first run only).
build_database()
con = get_connection()

def sql(query: str):
    """Run a query and return it as a DataFrame."""
    return con.execute(query).df()

## Schema
This exact text is what the LLM will receive to know which tables and columns exist.

In [ ]:
print(describe_schema())

## Basic checks

In [ ]:
sql("""
SELECT MIN(date) AS date_from, MAX(date) AS date_to,
       COUNT(*) AS n_rows,
       COUNT(DISTINCT store_nbr) AS n_stores,
       COUNT(DISTINCT family) AS n_families
FROM sales
""")

## Question 1: which families sell the most overall?

In [ ]:
sql("""
SELECT family, ROUND(SUM(sales)) AS total_sales
FROM sales
GROUP BY family
ORDER BY total_sales DESC
LIMIT 10
""")

## Question 2: which cities sold the most beverages in December 2016?

In [ ]:
sql("""
SELECT city, ROUND(SUM(sales)) AS sales
FROM sales_enriched
WHERE family = 'BEVERAGES'
  AND date BETWEEN DATE '2016-12-01' AND DATE '2016-12-31'
GROUP BY city
ORDER BY sales DESC
LIMIT 5
""")

## Question 3: how do school supplies sales change by month?
The hardest family in the previous project: the back-to-school peak shows up here.

In [ ]:
sql("""
SELECT DATE_TRUNC('month', date) AS month, ROUND(SUM(sales)) AS sales
FROM sales
WHERE family = 'SCHOOL AND OFFICE SUPPLIES'
GROUP BY month
ORDER BY month
""").plot(x="month", y="sales", figsize=(12, 4), title="SCHOOL AND OFFICE SUPPLIES by month");

## Question 4: how much do sales rise with promotions?

In [ ]:
sql("""
SELECT family,
       ROUND(AVG(sales) FILTER (WHERE onpromotion > 0), 1) AS avg_with_promo,
       ROUND(AVG(sales) FILTER (WHERE onpromotion = 0), 1) AS avg_without_promo,
       ROUND(avg_with_promo / NULLIF(avg_without_promo, 0), 2) AS ratio
FROM sales
WHERE date >= DATE '2016-01-01'
GROUP BY family
ORDER BY ratio DESC NULLS LAST
LIMIT 10
""")

## Notes for the agent

- Natural-language questions hide decisions: "December" of which year? "sold the most" in total units or daily average?
  The agent must either ask or state its assumption. Log any ambiguities found here.
- The promotion association is not causal: promotions are not assigned at random.

In [ ]:
con.close()